# Create Icechunk from Kerchunk

In [2]:
# get kerchunks
import gcsfs

fs = gcsfs.GCSFileSystem(token="anon")

files = fs.glob(
    "noaa-oar-cefi-regional-mom6/"
    "northwest_atlantic/full_domain/hindcast/monthly/regrid/"
    "r20250715/*.json"
)

kerchunk_jsons = [
    "https://storage.googleapis.com/" + f
    for f in files
    if f.endswith(".json")
]

for f in kerchunk_jsons[:3]:
    print(f)

# https://storage.googleapis.com/noaa-oar-cefi-regional-mom6//northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json
# https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json

https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json
https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/BHEAT.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json
https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/BMELT.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json


In [8]:
import json
from pathlib import Path
import gcsfs

fs = gcsfs.GCSFileSystem(token="anon")

files = fs.glob(
    "noaa-oar-cefi-regional-mom6/"
    "northwest_atlantic/full_domain/hindcast/monthly/regrid/"
    "r20250715/*.json"
)

out_dir = Path("patched_kerchunk_refs")
out_dir.mkdir(exist_ok=True)

patched_refs = []

for f in files:

    gcs_path = "gcs://" + f

    with fs.open(gcs_path, "r") as infile:
        refs = json.load(infile)

    # Rewrite chunk target URLs
    for k, v in refs["refs"].items():

        if isinstance(v, list) and isinstance(v[0], str):

            v[0] = v[0].replace(
                "gcs://noaa-oar-cefi-regional-mom6/",
                "https://storage.googleapis.com/noaa-oar-cefi-regional-mom6/",
            )

    out_path = out_dir / Path(f).name

    with open(out_path, "w") as outfile:
        json.dump(refs, outfile)

    patched_refs.append(out_path)

    print("Wrote:", out_path)

Wrote: patched_kerchunk_refs/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json
Wrote: patched_kerchunk_refs/BHEAT.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json
Wrote: patched_kerchunk_refs/BMELT.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json
Wrote: patched_kerchunk_refs/BSNK.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json
Wrote: patched_kerchunk_refs/CN.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json
Wrote: patched_kerchunk_refs/E2MELT.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json
Wrote: patched_kerchunk_refs/EXT.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json
Wrote: patched_kerchunk_refs/EXT_MAX.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json
Wrote: patched_kerchunk_refs/EXT_MIN.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json
Wrote: patched_kerchunk_refs/FRAZIL.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json
Wrote: patched_kerchunk_refs/Heat_PmE.nwa.full.hcast.monthly.r

KeyboardInterrupt: 

In [ ]:
patched_refs

In [13]:
from pathlib import Path
import obstore.store as obs

from virtualizarr import open_virtual_dataset
from virtualizarr.registry import ObjectStoreRegistry
from virtualizarr.parsers import KerchunkJSONParser

BASE = "https://storage.googleapis.com/noaa-oar-cefi-regional-mom6"

cwd = Path.cwd().resolve()
local_base = cwd.as_uri()

registry = ObjectStoreRegistry()

# Remote NetCDF chunk targets inside the patched refs
registry.register(BASE, obs.HTTPStore(BASE))

# Local patched Kerchunk JSON files
registry.register(local_base, obs.LocalStore(str(cwd)))

parser = KerchunkJSONParser()

patched_ref_paths = [
    str(Path("patched_kerchunk_refs") / Path(str(p)).name)
    for p in patched_refs
]

vsets = [
    open_virtual_dataset(
        ref,
        registry=registry,
        parser=parser,
    )
    for ref in patched_ref_paths
]


NotImplementedError: Reading inlined reference data is currently not supported.See https://github.com/zarr-developers/VirtualiZarr/issues/489

In [14]:
vds = open_virtual_dataset(
    patched_ref_paths[0],
    registry=registry,
    parser=KerchunkJSONParser(
        skip_variables=["lead", "member", "month", "init"]
    ),
)

print(vds)

NotImplementedError: Reading inlined reference data is currently not supported.See https://github.com/zarr-developers/VirtualiZarr/issues/489

In [22]:
import obstore.store as obs

from virtualizarr import open_virtual_dataset
from virtualizarr.registry import ObjectStoreRegistry
from virtualizarr.parsers import KerchunkJSONParser

BASE = "https://storage.googleapis.com/noaa-oar-cefi-regional-mom6"

kerchunk_jsons_https = [
    f.replace(
        "gcs://noaa-oar-cefi-regional-mom6",
        BASE,
    )
    for f in kerchunk_jsons
]

registry = ObjectStoreRegistry()

store = obs.HTTPStore(BASE)

registry.register(BASE, store)

vsets = [
    open_virtual_dataset(
        ref,
        registry=registry,
        parser=KerchunkJSONParser(),
    )
    for ref in kerchunk_jsons_https
]


/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


ValueError: paths in the manifest must be absolute posix paths or URIs, but got gcs://noaa-oar-cefi-regional-mom6/northwest_atlantic/full_domain/hindcast/monthly/regrid/r20250715/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.nc, and fs_root was not specified

In [ ]:


# Combine them, if needed
vds = xr.combine_by_coords(
    vsets,
    combine_attrs="override",
)

# Create Icechunk output repo
storage = icechunk.s3_storage(
    bucket="YOUR_OUTPUT_BUCKET",
    prefix="path/to/output.icechunk",
    region="us-west-2",
    from_env=True,
)

repo = icechunk.Repository.create(storage)

# Open repo authorizing the original public GCS NetCDF source
# The refs inside the kerchunk JSON point here.
virtual_chunk_container = icechunk.gcs_storage(
    bucket="noaa-oar-cefi-regional-mom6",
    prefix="",
    anonymous=True,
)

repo = icechunk.Repository.open(
    storage,
    authorize_virtual_chunk_access=[virtual_chunk_container],
)

session = repo.writable_session("main")

# Write virtual references into Icechunk
vds.virtualize.to_icechunk(session.store)

session.commit("Create Icechunk store from existing Kerchunk JSON refs")

In [15]:
import icechunk as ic

# Define your storage (local or cloud)
storage = ic.local_filesystem_storage(path='my_icechunk_repo')

# Create a new repository
repo = ic.Repository.create(storage)


  2026-05-27T23:00:52.334480Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:196



In [17]:
import virtualizarr as vz

# Open existing Kerchunk references as a virtual dataset
vds = vz.open_virtual_dataset(kerchunk_jsons[0])


TypeError: open_virtual_dataset() missing 2 required positional arguments: 'registry' and 'parser'

In [26]:
import json
from pathlib import Path
import icechunk as ic
import obstore
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import KerchunkJSONParser
import xarray as xr

# 1. Paths and setup
kerchunk_file = "patched_kerchunk_refs/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json"
temp_filtered_file = "patched_kerchunk_refs/filtered_data_only.json"

# 2. Extract and split the Kerchunk JSON file
with open(kerchunk_file, "r") as f:
    full_manifest = json.load(f)

refs = full_manifest.get("refs", full_manifest)

# Separate variables into "inline string data" and "remote chunk data"
inline_refs = {}
chunk_refs = {}

for key, val in refs.items():
    if isinstance(val, (str, bytes)) or (isinstance(val, list) and len(val) == 1 and isinstance(val[0], str) and not val[0].startswith("http")):
        inline_refs[key] = val
    else:
        chunk_refs[key] = val

# Save the clean chunk-only references to a temporary JSON file
filtered_manifest = {**full_manifest, "refs": chunk_refs}
with open(temp_filtered_file, "w") as f:
    json.dump(filtered_manifest, f)

# 3. Configure the registry for the local filtered JSON file
absolute_path = str(Path(temp_filtered_file).resolve())
absolute_url = f"file://{absolute_path}"

local_store = obstore.store.LocalStore()
registry = ObjectStoreRegistry({absolute_url: local_store})
parser = KerchunkJSONParser()

# 4. Open the filtered large chunk arrays via VirtualiZarr
with open_virtual_dataset(url=absolute_url, registry=registry, parser=parser) as vds:
    
    # 5. Set up Icechunk repository configuration
    config = ic.config.RepositoryConfig.default()
    target_data_prefix = "https://googleapis.com"
    
    container = ic.virtual.VirtualChunkContainer(
        url_prefix=target_data_prefix,
        store=ic.storage.http_store(url=target_data_prefix)
    )
    config.set_virtual_chunk_container(container)
    
    # 6. Create the local Icechunk repository
    ice_storage = ic.local_filesystem_storage(path="my_icechunk_repo")
    repo = ic.Repository.create(ice_storage, config)
    
    # 7. Write the large chunk references into Icechunk
    with repo.writable_session(branch="main") as session:
        vds.vz.to_icechunk(session.store)
        
        # 8. Manually inject the inline coordinate variables directly into Icechunk
        # This solves the missing metadata coordinate issue cleanly
        ice_zarr_store = session.store
        for key, value in inline_refs.items():
            # If it's a string metadata attribute or coordinate array text block
            if isinstance(value, str):
                ice_zarr_store[key] = value.encode('utf-8')
            else:
                ice_zarr_store[key] = value
                
        session.commit("Ingested HTTPS Kerchunk with isolated inline metadata values")

# Clean up the temporary file
Path(temp_filtered_file).unlink(missing_ok=True)
print("Icechunk repository created successfully with inline coordinates patched!")


TypeError: http_store() got an unexpected keyword argument 'url'

In [28]:
import json
from pathlib import Path
import icechunk as ic
import obstore
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import KerchunkJSONParser

# 1. Path to your locally patched JSON file
kerchunk_file = "patched_json/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json"
temp_filtered_file = "patched_json/filtered_data_only.json"

# 2. Extract and split the Kerchunk JSON file to remove inline coordinates
with open(kerchunk_file, "r") as f:
    full_manifest = json.load(f)

refs = full_manifest.get("refs", full_manifest)

inline_refs = {}
chunk_refs = {}

for key, val in refs.items():
    if isinstance(val, (str, bytes)) or (isinstance(val, list) and len(val) == 1 and isinstance(val, str) and not val.startswith("http")):
        inline_refs[key] = val
    else:
        chunk_refs[key] = val

filtered_manifest = {**full_manifest, "refs": chunk_refs}
with open(temp_filtered_file, "w") as f:
    json.dump(filtered_manifest, f)

# 3. Configure the registry for the local filtered JSON file
absolute_path = str(Path(temp_filtered_file).resolve())
absolute_url = f"file://{absolute_path}"

local_store = obstore.store.LocalStore()
registry = ObjectStoreRegistry({absolute_url: local_store})
parser = KerchunkJSONParser()

# 4. Open the filtered large chunk arrays via VirtualiZarr
with open_virtual_dataset(url=absolute_url, registry=registry, parser=parser) as vds:
    
    # 5. Set up Icechunk repository configuration
    config = ic.config.RepositoryConfig.default()
    
    # FIX: Provide a strict, absolute domain base ending in a / character
    target_data_base = "https://googleapis.com"
    
    container = ic.virtual.VirtualChunkContainer(
        url_prefix=target_data_base,
        store=ic.storage.http_store()
    )
    config.set_virtual_chunk_container(container)
    
    # 6. Create the local Icechunk repository
    ice_storage = ic.local_filesystem_storage(path="my_icechunk_repo")
    repo = ic.Repository.create(ice_storage, config)
    
    # 7. Write the large chunk references into Icechunk
    with repo.writable_session(branch="main") as session:
        vds.vz.to_icechunk(session.store)
        
        # 8. Manually inject the inline coordinate variables directly into Icechunk
        ice_zarr_store = session.store
        for key, value in inline_refs.items():
            if isinstance(value, str):
                ice_zarr_store[key] = value.encode('utf-8')
            else:
                ice_zarr_store[key] = value
                
        session.commit("Ingested HTTPS Kerchunk with isolated inline metadata values")

# Clean up the temporary file
Path(temp_filtered_file).unlink(missing_ok=True)
print("Icechunk repository created successfully with inline coordinates patched!")


ValueError: VirtualChunkContainer url_prefix must end in a / character

In [36]:
from pathlib import Path
import json
import icechunk as ic
import obstore.store as obs
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import KerchunkJSONParser

kerchunk_file = Path("patched_json/ALB.nwa.full.hcast.monthly.regrid.r20250715.199301-202312.json")
temp_filtered_file = Path("patched_json/filtered_data_only.json")

with open(kerchunk_file, "r") as f:
    full_manifest = json.load(f)

refs = full_manifest["refs"]

metadata_refs = {}
chunk_refs = {}
inline_data_refs = {}

for key, val in refs.items():
    # Always keep Zarr metadata
    if key.endswith(".zarray") or key.endswith(".zattrs") or key == ".zgroup":
        metadata_refs[key] = val

    # Keep external chunk byte-range refs
    elif isinstance(val, list) and len(val) == 3 and isinstance(val[0], str):
        chunk_refs[key] = val

    # Drop inline/base64/small literal chunks for now
    else:
        inline_data_refs[key] = val

filtered_manifest = {
    **full_manifest,
    "refs": {
        **metadata_refs,
        **chunk_refs,
    },
}

with open(temp_filtered_file, "w") as f:
    json.dump(filtered_manifest, f)

# Local manifest registry
local_dir = temp_filtered_file.parent.resolve()
local_base = local_dir.as_uri() + "/"
absolute_url = temp_filtered_file.resolve().as_uri()

registry = ObjectStoreRegistry()
registry.register(local_base, obs.LocalStore(str(local_dir)))

# Remote chunk registry
remote_base = "https://storage.googleapis.com/noaa-oar-cefi-regional-mom6"
registry.register(remote_base, obs.HTTPStore(remote_base))

parser = KerchunkJSONParser()

vds = open_virtual_dataset(
    url=absolute_url,
    registry=registry,
    parser=parser,
)

config = ic.config.RepositoryConfig.default()

container = ic.virtual.VirtualChunkContainer(
    url_prefix=remote_base,
    store=ic.storage.http_store(),
)

config.set_virtual_chunk_container(container)

ice_storage = ic.local_filesystem_storage(path="my_icechunk_repo")
repo = ic.Repository.create(ice_storage, config)

session = repo.writable_session(branch="main")
vds.vz.to_icechunk(session.store)
session.commit("Ingested Kerchunk virtual refs into Icechunk")

temp_filtered_file.unlink(missing_ok=True)
print("Icechunk repository created")


/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


ValueError: VirtualChunkContainer url_prefix must end in a / character

In [32]:
import icechunk as ic
import zarr

storage = ic.local_filesystem_storage("my_icechunk_repo")

repo = ic.Repository.open(storage)

session = repo.readonly_session(branch="main")

root = zarr.open_group(session.store, mode="r")

print(root.tree())

  2026-05-27T23:35:29.900253Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:196



/

In [33]:
print(list(root.array_keys())[:10])

[]


In [34]:
print(vds)

print(vds.data_vars)
print(vds.coords)
print(vds.sizes)

<xarray.Dataset> Size: 0B
Dimensions:  ()
Data variables:
    *empty*
Data variables:
    *empty*
Coordinates:
    *empty*
Frozen({})
